# Rubric Agent

This notebook walks through the three-stage rubric pipeline: extracting
requirements from policy text, refining with historical audit findings,
and producing evidence-backed verdicts against a project description.

## Imports

In [ ]:
from agentic_patterns.core.rubric import (
    RubricBuilder,
    RubricEvaluator,
    refine_with_history,
)
from agentic_patterns.core.vectordb import get_vector_db, MultiSourceRetriever
from agentic_patterns.core.vectordb.chunking import chunk_by_paragraphs
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance

## Stage 1 -- Build rubric from policy

Ingest a SOC 2 policy into a vector database, then extract requirement items (MUST / SHOULD / MAY).

In [ ]:
POLICY_TEXT = """\
Access Control Policy
All production systems MUST enforce role-based access control (RBAC).
User accounts MUST be reviewed quarterly and inactive accounts disabled within 30 days.
Privileged access SHOULD require multi-factor authentication at every login.

Encryption Policy
Data at rest MUST be encrypted using AES-256 or equivalent.
Data in transit MUST use TLS 1.2 or higher for all internal and external communications.
Encryption keys SHOULD be rotated annually and MAY be rotated more frequently for sensitive systems.

Logging and Monitoring Policy
All authentication events MUST be logged with timestamp, user ID, and outcome.
Administrative actions MUST be captured in an immutable audit trail.
Anomaly detection SHOULD be enabled on critical systems and alerts reviewed within 24 hours.

Incident Response Policy
A documented incident response plan MUST be maintained and tested annually.
Security incidents MUST be reported to the security team within one hour of detection.
Post-incident reviews SHOULD be completed within five business days.
"""

policy_index = get_vector_db("rubric_demo_policy")
if policy_index.count() == 0:
    chunks = chunk_by_paragraphs(
        POLICY_TEXT, DocumentProvenance(source="soc2_policy"), min_lines=1
    )
    policy_index.ingest(chunks)

In [ ]:
builder = RubricBuilder()
rubric = await builder.build_from_policy(policy_index, rubric_name="soc2_demo")

In [ ]:
print(f"Rubric: {rubric.rubric_id}  ({len(rubric.items)} items)\n")
for item in rubric.items:
    print(f"[{item.requirement_level.value}] {item.title}  (weight={item.weight})")

## Stage 2 -- Refine with historical findings

Ingest mock audit findings, bump weights for recurring items, and promote new concerns.

In [ ]:
AUDIT_FINDINGS_TEXT = """\
Q3 Audit Finding: Three service accounts with production access had not been reviewed in over six months.
Q3 Audit Finding: Two internal microservices communicated over plain HTTP instead of TLS.
Q3 Audit Finding: Admin console actions were logged but logs lacked immutability guarantees.

Q1 Audit Finding: Quarterly access review was completed 15 days late for the payments team.
Q1 Audit Finding: Encryption key rotation had not occurred for the analytics database in 18 months.

Q4 Audit Finding: Incident response plan existed but had not been tested since initial creation two years ago.
Q4 Audit Finding: Two critical alerts from the anomaly detection system went unacknowledged for 48 hours.
"""

history_index = get_vector_db("rubric_demo_history")
if history_index.count() == 0:
    chunks = chunk_by_paragraphs(
        AUDIT_FINDINGS_TEXT, DocumentProvenance(source="audit_findings"), min_lines=1
    )
    history_index.ingest(chunks)

In [ ]:
rubric_v2 = await refine_with_history(rubric, history_index, policy_index)

In [ ]:
print(
    f"Rubric v{rubric_v2.provenance.get('version', '?')}  ({len(rubric_v2.items)} items)\n"
)
for item in rubric_v2.items:
    print(f"[{item.requirement_level.value}] {item.title}  (weight={item.weight})")

## Stage 3 -- Evidence-backed assessment

Ingest the project description, then evaluate each rubric item against the project, policy, and audit history.

In [ ]:
PROJECT_DESCRIPTION_TEXT = """\
Project Aurora -- Security Posture Summary

Aurora enforces RBAC via AWS IAM with quarterly access reviews automated through a custom script.
MFA is required for all human users but not for CI/CD service accounts.

All databases use AES-256 encryption at rest. Internal service-to-service traffic uses mTLS.
Encryption key rotation is handled by AWS KMS with a 365-day rotation policy.

Authentication events are logged to CloudWatch with structured JSON entries.
Admin actions are captured but stored in the same mutable log stream as application logs.

Aurora has a documented incident response runbook. It was last tested eight months ago.
Anomaly detection is not currently enabled; the team relies on manual dashboard reviews.
"""

project_index = get_vector_db("rubric_demo_project")
if project_index.count() == 0:
    chunks = chunk_by_paragraphs(
        PROJECT_DESCRIPTION_TEXT, DocumentProvenance(source="project_aurora"), min_lines=1
    )
    project_index.ingest(chunks)

In [ ]:
retriever = MultiSourceRetriever(
    {
        "policy": policy_index,
        "history": history_index,
        "project": project_index,
    }
)
evaluator = RubricEvaluator()
verdicts = await evaluator.evaluate(rubric_v2, retriever)

In [ ]:
for v in verdicts:
    print(f"[{v.status.value}] {v.item_id}")
    print(f"  Rationale: {v.rationale[:120]}")
    if v.missing_evidence:
        print(f"  Missing: {', '.join(v.missing_evidence)}")
    print()